# Cortina Generator — Test Notebook
Tests each layer: tanda summarizer → prompt crafter → Lyria audio → full generate_cortina.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from atdj.config import GEMINI_API_KEY
print('GEMINI_API_KEY set:', bool(GEMINI_API_KEY))

GEMINI_API_KEY set: True


## 1 — Test `_summarize_tanda`

In [2]:
from atdj.cortina.generator import _summarize_tanda

mock_tracks = [
    {'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy': 0.6},
    {'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy': 0.7},
    {'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy': 0.65},
]

summary = _summarize_tanda(mock_tracks)
print(summary)

{'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy_label': 'moderate'}


## 2 — Test `_craft_music_prompt` (uses Gemini LLM)

In [3]:
from atdj.cortina.generator import _craft_music_prompt

prev = _summarize_tanda(mock_tracks)

# Closing cortina (next_style=None)
closing_prompt = _craft_music_prompt(prev, next_style=None)
print('=== Closing cortina prompt ===')
print(closing_prompt)

2026-05-03 19:50:56.218 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 19:50:56.218 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-05-03 19:50:56.218 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 19:50:56.218 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 19:50:56.218 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 19:50:56.218 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

=== Closing cortina prompt ===
Generate a 25-second closing cortina with a soft, contemplative mood, featuring a gentle piano melody accompanied by sustained, airy flute textures. The music should slowly dissipate into a peaceful, neutral ambience, clearly devoid of any tango rhythm or bandoneon.


## 3 — Test `_call_lyria` (generates audio)

In [4]:
from pathlib import Path
from atdj.cortina.generator import _call_lyria

out_dir = Path('../data/cortinas/generated')
out_dir.mkdir(parents=True, exist_ok=True)

audio_path = _call_lyria(closing_prompt, out_dir / 'test_cortina', GEMINI_API_KEY)
print('Audio saved to:', audio_path)
print('File size:', audio_path.stat().st_size, 'bytes')

Audio saved to: ../data/cortinas/generated/test_cortina.mp3
File size: 712007 bytes


## 4 — Play the audio

In [5]:
from IPython.display import Audio
Audio(str(audio_path))

## 5 — Test full `generate_cortina`

In [6]:
from atdj.cortina.generator import generate_cortina

result = generate_cortina(
    prev_tracks=mock_tracks,
    next_style=None,
    output_dir=out_dir,
    api_key=GEMINI_API_KEY,
)
print(result)
Audio(result['file_path'])

2026-05-03 20:02:34.742 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 20:02:34.743 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 20:02:34.744 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 20:02:34.744 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 20:02:34.744 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


{'type': 'cortina', 'title': 'Closing Cortina (tango)', 'file_path': '../data/cortinas/generated/cortina_be9abf33.mp3', 'duration': '0:25', 'source': 'generated', 'music_prompt': 'Compose a 25-second closing cortina, distinctly non-tango, featuring a peaceful, flowing composition. Utilize piano, sustained strings, and a soft clarinet to emphasize sustained harmonies and a slow, legato melodic line. This piece should gently fade, signaling the end of the set.'}
